# Deep Ensembles — Toy Regression Experiment

This notebook reproduces the **1-D toy regression experiment** from:

> *"Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles"*  
> Lakshminarayanan, Pritzel & Blundell (2017), NeurIPS.

We implement **three experimental protocols** that progressively improve uncertainty quantification:

| Protocol | Model | Loss | Uncertainty |
|----------|-------|------|-------------|
| 1 | Single NN | MSE | None (point prediction) |
| 2 | Single NN | NLL | Aleatoric only |
| 3 | Ensemble of $M = 5$ NNs | NLL | Aleatoric + Epistemic |

**Framework:** JAX (following ASI 2026 course Tutorial 2).

---
## Setup

Import all required libraries.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

# Render plots inline
%matplotlib inline

---
## Configuration

All hyper-parameters and constants used throughout the experiment.

In [ ]:
SEED = 42
N_TRAIN = 20          # number of training points
X_RANGE = (-4, 4)     # training x range
X_TEST_RANGE = (-6, 6)  # test x range (wider, to see OOD behavior)
N_TEST = 200          # number of test points
HIDDEN_DIM = 50       # hidden layer size
LR = 0.01             # learning rate
N_EPOCHS = 5000       # training epochs
M_ENSEMBLE = 5        # ensemble size

# Output directory for figures
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Matplotlib style
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

---
## Data Generation

We use the toy regression setup from the paper. The ground-truth function is a simple cubic:

$$y = x^3 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 9)$$

Training points are sampled uniformly from $x \in [-4, 4]$, while test points span a wider range $x \in [-6, 6]$ to evaluate **out-of-distribution (OOD)** behavior — a key motivation for uncertainty estimation.

In [ ]:
def generate_data(key, n=N_TRAIN, x_range=X_RANGE, noise_std=3.0):
    """Generate toy regression data: y = x^3 + epsilon, epsilon ~ N(0, 9)."""
    key_x, key_eps = random.split(key)
    x = random.uniform(key_x, shape=(n, 1), minval=x_range[0], maxval=x_range[1])
    eps = noise_std * random.normal(key_eps, shape=(n, 1))
    y = x ** 3 + eps
    return x, y

---
## Neural Network Architecture

We use a simple **Multi-Layer Perceptron (MLP)** with:

- **1 hidden layer** with 50 units and **ReLU** activation
- **2 output units** representing $\mu(x)$ (predicted mean) and $\log \sigma^2(x)$ (log predicted variance)
- **He initialization** for weights: $W \sim \mathcal{N}\!\left(0, \frac{2}{n_{\text{in}}}\right)$

The predicted variance is obtained by passing the raw log-variance output through a **softplus** function to ensure positivity:

$$\sigma^2(x) = \text{softplus}\!\big(\log\hat{\sigma}^2(x)\big) + \epsilon$$

where $\text{softplus}(z) = \log(1 + e^z)$ and $\epsilon = 10^{-6}$ prevents numerical issues.

In [ ]:
def init_params(key, input_dim=1, hidden_dim=HIDDEN_DIM, output_dim=2):
    """
    Initialize a single-hidden-layer MLP.
    Output dim = 2 for NLL (mu, log_var) or 1 for MSE (mu only).
    Uses He initialization for weights.
    """
    k1, k2 = random.split(key)
    params = {
        'W1': random.normal(k1, (input_dim, hidden_dim)) * jnp.sqrt(2.0 / input_dim),
        'b1': jnp.zeros(hidden_dim),
        'W2': random.normal(k2, (hidden_dim, output_dim)) * jnp.sqrt(2.0 / hidden_dim),
        'b2': jnp.zeros(output_dim),
    }
    return params

In [ ]:
def forward(params, x):
    """Forward pass: input -> hidden (ReLU) -> output."""
    h = jnp.maximum(0, x @ params['W1'] + params['b1'])  # ReLU
    out = h @ params['W2'] + params['b2']
    return out


def softplus(x):
    """Numerically stable softplus: log(1 + exp(x))."""
    return jnp.logaddexp(x, 0.0)


def predict_gaussian(params, x):
    """
    Predict mean and variance from network output.
    Returns mu, var (both shape (n, 1)).
    """
    out = forward(params, x)
    mu = out[:, 0:1]
    log_var = out[:, 1:2]
    var = softplus(log_var) + 1e-6  # enforce positivity
    return mu, var

---
## Loss Functions

### Mean Squared Error (MSE) — Protocol 1

The standard MSE loss provides only a **point estimate** with no uncertainty:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N} \big(y_i - \mu_\theta(x_i)\big)^2$$

### Negative Log-Likelihood (NLL) — Protocols 2 & 3

Assuming the network outputs a Gaussian $\mathcal{N}\!\big(\mu_\theta(x),\, \sigma^2_\theta(x)\big)$, the NLL loss (**Eq. 2** from the paper) is:

$$\mathcal{L}_{\text{NLL}} = \frac{1}{N}\sum_{i=1}^{N} \left[ \frac{1}{2}\log \sigma^2_\theta(x_i) + \frac{\big(y_i - \mu_\theta(x_i)\big)^2}{2\,\sigma^2_\theta(x_i)} \right]$$

This loss simultaneously learns the mean **and** the variance, providing a measure of **aleatoric** (data) uncertainty.

In [ ]:
def mse_loss(params, x, y):
    """Mean Squared Error loss (Protocol 1)."""
    out = forward(params, x)
    mu = out[:, 0:1]  # only use first output
    return jnp.mean((y - mu) ** 2)


def nll_loss(params, x, y):
    """
    Negative Log-Likelihood loss for Gaussian output (Protocol 2 & 3).
    Eq. (2) from the report:
      L = mean[ 0.5 * log(var) + 0.5 * (y - mu)^2 / var ]
    """
    mu, var = predict_gaussian(params, x)
    return jnp.mean(0.5 * jnp.log(var) + 0.5 * (y - mu) ** 2 / var)

---
## Adam Optimizer

We implement the **Adam** optimizer from scratch (following the JAX tutorial style). Adam maintains exponential moving averages of the gradient (first moment $m$) and squared gradient (second moment $v$), with bias correction:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
def init_adam(params, lr=LR, beta1=0.9, beta2=0.999, eps=1e-8):
    """Initialize Adam optimizer state."""
    state = {
        'm': jax.tree.map(jnp.zeros_like, params),     # first moment
        'v': jax.tree.map(jnp.zeros_like, params),     # second moment
        't': 0,
        'lr': lr,
        'beta1': beta1,
        'beta2': beta2,
        'eps': eps,
    }
    return state


def adam_step(params, grads, state):
    """Perform one Adam update step."""
    t = state['t'] + 1
    lr, beta1, beta2, eps = state['lr'], state['beta1'], state['beta2'], state['eps']

    m = jax.tree.map(lambda m, g: beta1 * m + (1 - beta1) * g, state['m'], grads)
    v = jax.tree.map(lambda v, g: beta2 * v + (1 - beta2) * g ** 2, state['v'], grads)

    # Bias correction
    m_hat = jax.tree.map(lambda m: m / (1 - beta1 ** t), m)
    v_hat = jax.tree.map(lambda v: v / (1 - beta2 ** t), v)

    # Update params
    params = jax.tree.map(lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps),
                          params, m_hat, v_hat)

    state = {**state, 'm': m, 'v': v, 't': t}
    return params, state

---
## Training Loop

A generic training function that accepts any loss function (MSE or NLL). It uses JAX's `jit` and `grad` for efficient automatic differentiation and compilation.

In [ ]:
def train(params, x, y, loss_fn, n_epochs=N_EPOCHS, lr=LR, verbose=True):
    """Train a network with the given loss function."""
    opt_state = init_adam(params, lr=lr)
    grad_fn = jit(grad(loss_fn))
    loss_fn_jit = jit(loss_fn)

    losses = []
    for epoch in range(n_epochs):
        grads = grad_fn(params, x, y)
        params, opt_state = adam_step(params, grads, opt_state)

        if epoch % 1000 == 0 or epoch == n_epochs - 1:
            loss_val = float(loss_fn_jit(params, x, y))
            losses.append(loss_val)
            if verbose:
                print(f"  Epoch {epoch:5d}/{n_epochs} | Loss: {loss_val:.4f}")

    return params, losses

---
## Ensemble Prediction

The key idea of **Deep Ensembles** is to train $M$ networks independently (with different random initializations) and combine their predictions as a **mixture of Gaussians**.

From **Eq. (5)** in the paper, the ensemble predictive distribution is approximated as:

$$\mu_*(x) = \frac{1}{M} \sum_{m=1}^{M} \mu_m(x)$$

$$\sigma^2_*(x) = \frac{1}{M} \sum_{m=1}^{M} \left[ \sigma^2_m(x) + \mu_m(x)^2 \right] - \mu_*(x)^2$$

The total variance naturally decomposes into two components:

- **Aleatoric uncertainty** (irreducible noise in the data): $\quad \sigma^2_{\text{alea}} = \frac{1}{M}\sum_m \sigma^2_m(x)$

- **Epistemic uncertainty** (model disagreement): $\quad \sigma^2_{\text{epis}} = \frac{1}{M}\sum_m \mu_m(x)^2 - \mu_*(x)^2$

Epistemic uncertainty is expected to be **high outside the training range** (OOD regions) and **low where data is abundant**.

In [ ]:
def ensemble_predict(all_params, x_test):
    """
    Combine M ensemble members using Eq. (5):
      mu* = (1/M) sum_m mu_m
      var* = (1/M) sum_m [var_m + mu_m^2] - mu*^2
    
    Returns mu_star, var_star, aleatoric, epistemic.
    """
    M = len(all_params)
    mus = []
    vars_ = []

    for params in all_params:
        mu, var = predict_gaussian(params, x_test)
        mus.append(mu)
        vars_.append(var)

    mus = jnp.stack(mus, axis=0)       # (M, N, 1)
    vars_ = jnp.stack(vars_, axis=0)   # (M, N, 1)

    # Ensemble mean
    mu_star = jnp.mean(mus, axis=0)    # (N, 1)

    # Total variance (Eq. 5)
    var_star = jnp.mean(vars_ + mus ** 2, axis=0) - mu_star ** 2

    # Decomposition
    aleatoric = jnp.mean(vars_, axis=0)               # average predicted noise
    epistemic = jnp.mean(mus ** 2, axis=0) - mu_star ** 2  # variance of means

    return mu_star, var_star, aleatoric, epistemic

---
## Visualization Helpers

Plotting functions for each protocol and the combined/decomposition figures.

In [ ]:
# Color palette
COLOR_DATA = '#2d3436'
COLOR_MEAN = '#0984e3'
COLOR_BAND = '#74b9ff'
COLOR_TRUE = '#636e72'
COLOR_ALEATORIC = '#00b894'
COLOR_EPISTEMIC = '#e17055'


def plot_true_function(ax, x_test):
    """Plot the ground truth y = x^3."""
    ax.plot(x_test, x_test ** 3, '--', color=COLOR_TRUE, alpha=0.5,
            linewidth=1.2, label=r'True $y = x^3$')


def plot_single_mse(ax, params, x_train, y_train, x_test):
    """Plot Protocol 1: Single NN + MSE."""
    out = forward(params, x_test)
    mu = out[:, 0]

    plot_true_function(ax, x_test.squeeze())
    ax.scatter(x_train.squeeze(), y_train.squeeze(), c=COLOR_DATA,
               s=30, zorder=5, label='Training data', edgecolors='white', linewidth=0.5)
    ax.plot(x_test.squeeze(), mu, color=COLOR_MEAN, linewidth=2, label=r'Predicted $\mu(x)$')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('(a) Single NN — MSE Loss')
    ax.legend(loc='upper left', framealpha=0.9)
    ax.set_ylim(-200, 200)


def plot_single_nll(ax, params, x_train, y_train, x_test):
    """Plot Protocol 2: Single NN + NLL."""
    mu, var = predict_gaussian(params, x_test)
    mu = mu.squeeze()
    std = jnp.sqrt(var).squeeze()
    x_t = x_test.squeeze()

    plot_true_function(ax, x_t)
    ax.fill_between(x_t, mu - 3 * std, mu + 3 * std, alpha=0.25,
                    color=COLOR_BAND, label=r'$\mu \pm 3\sigma$')
    ax.scatter(x_train.squeeze(), y_train.squeeze(), c=COLOR_DATA,
               s=30, zorder=5, label='Training data', edgecolors='white', linewidth=0.5)
    ax.plot(x_t, mu, color=COLOR_MEAN, linewidth=2, label=r'Predicted $\mu(x)$')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('(b) Single NN — NLL Loss')
    ax.legend(loc='upper left', framealpha=0.9)
    ax.set_ylim(-200, 200)


def plot_ensemble(ax, all_params, x_train, y_train, x_test, show_decomposition=False):
    """Plot Protocol 3: Ensemble of 5 NNs + NLL."""
    mu_star, var_star, aleatoric, epistemic = ensemble_predict(all_params, x_test)
    mu = mu_star.squeeze()
    std = jnp.sqrt(var_star).squeeze()
    x_t = x_test.squeeze()

    plot_true_function(ax, x_t)

    if show_decomposition:
        std_alea = jnp.sqrt(aleatoric).squeeze()
        std_epis = jnp.sqrt(epistemic).squeeze()
        ax.fill_between(x_t, mu - 3 * std, mu + 3 * std, alpha=0.15,
                        color=COLOR_BAND, label=r'Total $\mu \pm 3\sigma$')
        ax.fill_between(x_t, mu - 3 * std_alea, mu + 3 * std_alea, alpha=0.25,
                        color=COLOR_ALEATORIC, label='Aleatoric')
        ax.fill_between(x_t, mu - 3 * std_epis, mu + 3 * std_epis, alpha=0.25,
                        color=COLOR_EPISTEMIC, label='Epistemic')
    else:
        ax.fill_between(x_t, mu - 3 * std, mu + 3 * std, alpha=0.25,
                        color=COLOR_BAND, label=r'$\mu \pm 3\sigma$')

    # Plot individual ensemble members (faint)
    for i, params in enumerate(all_params):
        mu_i, _ = predict_gaussian(params, x_test)
        label = 'Individual NNs' if i == 0 else None
        ax.plot(x_t, mu_i.squeeze(), color=COLOR_MEAN, alpha=0.15,
                linewidth=0.8, label=label)

    ax.scatter(x_train.squeeze(), y_train.squeeze(), c=COLOR_DATA,
               s=30, zorder=5, label='Training data', edgecolors='white', linewidth=0.5)
    ax.plot(x_t, mu, color=COLOR_MEAN, linewidth=2, label=r'Ensemble $\mu_*(x)$')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('(c) Deep Ensemble (M=5) — NLL Loss')
    ax.legend(loc='upper left', framealpha=0.9, fontsize=8)
    ax.set_ylim(-200, 200)

---
## Generate Training and Test Data

In [ ]:
key = random.PRNGKey(SEED)
key, data_key = random.split(key)
x_train, y_train = generate_data(data_key)
x_test = jnp.linspace(X_TEST_RANGE[0], X_TEST_RANGE[1], N_TEST).reshape(-1, 1)

print(f"Training data: {N_TRAIN} points in [{X_RANGE[0]}, {X_RANGE[1]}]")
print(f"Test range: [{X_TEST_RANGE[0]}, {X_TEST_RANGE[1]}] ({N_TEST} points)")

---
# Run Experiments

## Protocol 1: Single NN + MSE

A single neural network trained with **Mean Squared Error** loss. This is the simplest baseline — the network learns only a point estimate $\mu(x)$ with **no uncertainty quantification**. The model cannot express how confident it is in its predictions, especially in out-of-distribution regions.

In [ ]:
print("Protocol 1: Single NN with MSE Loss")
print("-" * 40)
key, init_key = random.split(key)
params_mse = init_params(init_key, output_dim=2)  # 2 outputs but only use first
params_mse, losses_mse = train(params_mse, x_train, y_train, mse_loss)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(6, 4.5))
plot_single_mse(ax1, params_mse, x_train, y_train, x_test)
ax1.set_title('Single NN — MSE Loss')
plt.tight_layout()
fig1.savefig(FIG_DIR / 'single_nn_mse.pdf')
fig1.savefig(FIG_DIR / 'single_nn_mse.png')
plt.show()

## Protocol 2: Single NN + NLL

A single neural network trained with **Negative Log-Likelihood** loss. The network now outputs both $\mu(x)$ and $\sigma^2(x)$, learning to capture **aleatoric uncertainty** (irreducible noise in the data). However, with a single network, the model still cannot capture **epistemic uncertainty** — it cannot express when it is uncertain due to lack of data.

In [ ]:
print("Protocol 2: Single NN with NLL Loss")
print("-" * 40)
key, init_key = random.split(key)
params_nll = init_params(init_key, output_dim=2)
params_nll, losses_nll = train(params_nll, x_train, y_train, nll_loss)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 4.5))
plot_single_nll(ax2, params_nll, x_train, y_train, x_test)
ax2.set_title('Single NN — NLL Loss')
plt.tight_layout()
fig2.savefig(FIG_DIR / 'single_nn_nll.pdf')
fig2.savefig(FIG_DIR / 'single_nn_nll.png')
plt.show()

## Protocol 3: Ensemble of $M=5$ NNs + NLL

An ensemble of $M=5$ independently trained neural networks, each with NLL loss but **different random initializations**. The diversity arising from random initialization is sufficient to capture **epistemic uncertainty** through the disagreement between ensemble members.

This is the **Deep Ensembles** approach proposed by Lakshminarayanan et al. (2017) — simple, parallelizable, and empirically well-calibrated.

In [ ]:
print(f"Protocol 3: Ensemble of M={M_ENSEMBLE} NNs with NLL Loss")
print("-" * 40)
ensemble_params = []
for m in range(M_ENSEMBLE):
    print(f"\n  --- Ensemble member {m + 1}/{M_ENSEMBLE} ---")
    key, init_key = random.split(key)
    params_m = init_params(init_key, output_dim=2)
    params_m, _ = train(params_m, x_train, y_train, nll_loss)
    ensemble_params.append(params_m)

In [ ]:
fig3, ax3 = plt.subplots(figsize=(6, 4.5))
plot_ensemble(ax3, ensemble_params, x_train, y_train, x_test)
ax3.set_title(f'Deep Ensemble (M={M_ENSEMBLE}) — NLL Loss')
plt.tight_layout()
fig3.savefig(FIG_DIR / 'ensemble_nll.pdf')
fig3.savefig(FIG_DIR / 'ensemble_nll.png')
plt.show()

---
## Combined Results

The three-panel comparison figure below highlights the progressive improvement in uncertainty estimation across protocols. Notice how:

1. **Protocol 1 (MSE)**: No uncertainty bands — the model is equally "confident" everywhere, even far from training data.
2. **Protocol 2 (NLL)**: Uncertainty bands appear, but a single network may not grow uncertainty sufficiently in OOD regions.
3. **Protocol 3 (Ensemble)**: Well-calibrated uncertainty that grows appropriately outside the training range.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
plot_single_mse(axes[0], params_mse, x_train, y_train, x_test)
plot_single_nll(axes[1], params_nll, x_train, y_train, x_test)
plot_ensemble(axes[2], ensemble_params, x_train, y_train, x_test)
axes[1].set_ylabel('')
axes[2].set_ylabel('')
fig.suptitle('Toy Regression: Deep Ensembles vs. Single Networks',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / 'toy_regression_combined.pdf')
fig.savefig(FIG_DIR / 'toy_regression_combined.png')
plt.show()

---
## Uncertainty Decomposition

The Deep Ensemble framework allows us to decompose the total predictive uncertainty into its two components:

- **Aleatoric uncertainty** (green band): captures inherent noise in the data — roughly constant across the input domain since $\varepsilon \sim \mathcal{N}(0, 9)$.
- **Epistemic uncertainty** (orange band): captures model uncertainty due to limited data — grows in out-of-distribution regions ($|x| > 4$) where the model has never seen training examples.

In [ ]:
fig4, ax4 = plt.subplots(figsize=(7, 5))
plot_ensemble(ax4, ensemble_params, x_train, y_train, x_test, show_decomposition=True)
ax4.set_title(f'Uncertainty Decomposition — Deep Ensemble (M={M_ENSEMBLE})')
plt.tight_layout()
fig4.savefig(FIG_DIR / 'uncertainty_decomposition.pdf')
fig4.savefig(FIG_DIR / 'uncertainty_decomposition.png')
plt.show()

---
## Loss Curves

Training loss over epochs for MSE and NLL losses. Note that the two losses are on different scales (MSE measures squared pixel error, NLL measures negative log-probability), so their magnitudes are not directly comparable.

In [ ]:
fig5, ax5 = plt.subplots(figsize=(6, 4))
epochs_log = list(range(0, N_EPOCHS, 1000)) + [N_EPOCHS - 1]
ax5.plot(epochs_log, losses_mse, 'o-', label='MSE Loss', color='#e74c3c', markersize=4)
ax5.plot(epochs_log, losses_nll, 's-', label='NLL Loss', color='#3498db', markersize=4)
ax5.set_xlabel('Epoch')
ax5.set_ylabel('Loss')
ax5.set_title('Training Loss Curves')
ax5.legend()
ax5.grid(True, alpha=0.3)
plt.tight_layout()
fig5.savefig(FIG_DIR / 'loss_curves.pdf')
fig5.savefig(FIG_DIR / 'loss_curves.png')
plt.show()

---

**End of notebook.** All figures have been saved to the `figures/` directory and displayed inline above.